# 0.01 Clinical audit: why OS/PFS needed cleaning

Run `make clean-survival` first. Findings are written up in `references/survival_cleaning.md`.

In [1]:
import pandas as pd
from ucsd_pfs.config import (
    CLINICAL_FILES,
    OS_MATCH_TOL_DAYS,
    PROCESSED_SURVIVAL,
    PROCESSED_TIMEPOINTS,
    RAW_DATA_DIR,
)
from ucsd_pfs.dataset import audit_version_permutation, clean_clinical, read_clinical_raw

v1 = clean_clinical(read_clinical_raw(RAW_DATA_DIR / CLINICAL_FILES["v1"]))
v3 = clean_clinical(read_clinical_raw(RAW_DATA_DIR / CLINICAL_FILES["v3"]))
tp = pd.read_parquet(PROCESSED_TIMEPOINTS)
surv = pd.read_parquet(PROCESSED_SURVIVAL)

2026-10-28 12:13:17.272 | DEBUG    | ucsd_pfs.config:<module>:40 - PROJ_ROOT is /home/valentin/workspaces/ucsd


## 1. v3 is a row permutation of v1
Each v3 row matches exactly one v1 row on all date offsets. Every column moved with the dates except `id`, `brats_id` and `age`.

In [2]:
audit_version_permutation(v1, v3).round(2)

2026-10-28 12:13:17.976 | INFO     | ucsd_pfs.dataset:audit_version_permutation:155 - v3 rows matched to a v1 row by dates: 243/243 (2 ambiguous)


,column,unmoved,moved_with_dates
0,id,1.00,0.00
1,patient_id,1.00,0.00
2,tp,1.00,0.57
3,brats_id,1.00,0.00
4,field_strength,1.00,1.00
5,manufacturer,1.00,1.00
6,scanner_model,0.74,1.00
7,age,1.00,0.02
8,sex,0.57,0.98
9,race,0.64,0.99


## 2. Which OS agrees with the dates?
OS is defined as initial surgery to death, so it should equal `d_death - d_initial_surgery` on the same row.

In [3]:
tol = OS_MATCH_TOL_DAYS
v1i, v3i = v1.set_index("id"), v3.set_index("id").loc[v1["id"]]


def agree(os, dates):
    calc = dates["d_death"] - dates["d_initial_surgery"]
    m = os.notna() & calc.notna()
    return f"{((os - calc).abs() <= tol)[m].sum()}/{m.sum()}"


pd.Series(
    {
        "v1 OS vs v1 dates": agree(v1i["os_days_provided"], v1i),
        "v3 OS vs v3 dates": agree(v3i["os_days_provided"], v3i),
        "v1 OS vs v3 dates (same ID)": agree(v1i["os_days_provided"], v3i.loc[v1i.index]),
    },
    name="rows agreeing",
)

v1 OS vs v1 dates               0/61
v3 OS vs v3 dates               0/61
v1 OS vs v3 dates (same ID)    60/62
Name: rows agreeing, dtype: object

## 3. With v1 OS/PFS on v3 rows, every OS row is a scan 0–90 days after surgery
This matches the paper's selection rule for the survival subset.

In [4]:
rows = tp[tp["os_days_provided"].notna() | tp["pfs_days_provided"].notna()]
v3rows = tp[tp["v3_os_days_provided"].notna() | tp["v3_pfs_days_provided"].notna()]
pd.DataFrame(
    {
        "repaired (v1 OS)": rows["days_since_surgery"].describe(),
        "unrepaired (v3 OS)": v3rows["days_since_surgery"].describe(),
    }
).round(1)

,repaired (v1 OS),unrepaired (v3 OS)
count,90.0,84.0
mean,36.1,324.8
std,23.2,486.6
min,-24.0,6.0
25%,21.0,42.8
50%,28.0,181.0
75%,47.8,358.8
max,90.0,2657.0


## 4. Cleaned survival table and remaining QC flags

In [5]:
print(surv["os_days_source"].value_counts())
flags = surv["qc_flags"].str.split(";").explode()
flags[flags != ""].value_counts()

os_days_source
provided_v1         81
recomputed_dates    14
none                 1
Name: count, dtype: int64


qc_flags
no_surgery_date          20
os_missing               15
pfs_missing              14
no_death_date             7
pfs_eq_os                 3
timeline_inconsistent     2
os_mismatch_dates         2
pfs_negative              1
Name: count, dtype: int64

In [6]:
surv[surv["qc_flags"].str.contains("mismatch|inconsistent|negative|eq_os")][
    [
        "cohort_tp_id",
        "days_since_surgery",
        "os_days_provided",
        "os_days_recomputed",
        "pfs_days",
        "qc_flags",
    ]
]

,cohort_tp_id,days_since_surgery,os_days_provided,os_days_recomputed,pfs_days,qc_flags
0,UCSD-PTGBM-0004_01,25.0,1202.0,1202.0,619.0,timeline_inconsistent
20,UCSD-PTGBM-0061_01,11.0,82.0,NaN,-8.0,pfs_negative;no_surgery_date
32,UCSD-PTGBM-0075_01,-0.0,246.0,231.0,156.0,os_mismatch_dates
55,UCSD-PTGBM-0098_01,16.0,167.0,NaN,167.0,pfs_eq_os;no_surgery_date
64,UCSD-PTGBM-0107_01,52.0,364.0,399.0,168.0,os_mismatch_dates
78,UCSD-PTGBM-0136_01,27.0,90.0,90.0,90.0,pfs_eq_os
81,UCSD-PTGBM-0142_01,89.0,215.0,NaN,215.0,pfs_eq_os;no_surgery_date
83,UCSD-PTGBM-0145_01,70.0,439.0,439.0,207.0,timeline_inconsistent
